# Real printed-label DTU-Net/Tsimplex training

This notebook trains the paper authors' DTU-Net/Tsimplex path for **2,000 optimizer
steps using 40 registered normal photographs from eight physical labels**. It is the
training stage of the proposed printed-label application. It does not evaluate defects
and does not claim that the paper's numerical results were reproduced.

Attach `printed_label_train_v1.zip`, select a Kaggle GPU, enable Internet for the
pinned author source download, and run all cells. Download both output files at the end.


In [ ]:
from pathlib import Path
import importlib.util, json, os, random, shutil, subprocess, sys, time, zipfile

import numpy as np
import torch
from PIL import Image, ImageEnhance, ImageOps

assert Path('/kaggle/input').is_dir(), 'Run this notebook on Kaggle.'
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator first.'

KAGGLE_INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working/labelinspect')
WORK.mkdir(parents=True, exist_ok=True)
OUTPUT = WORK/'printed_label_training'
OUTPUT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = WORK/'checkpoints'/'printed_label'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def find_dataset(search_root):
    candidates=[]
    for path in search_root.rglob('printed_label_train_v1'):
        if path.is_dir() and (path/'train'/'normal').is_dir():
            candidates.append(path)
    return sorted(set(candidates))

dataset_candidates=find_dataset(KAGGLE_INPUT)
if not dataset_candidates:
    matching=[]
    for archive_path in KAGGLE_INPUT.rglob('*.zip'):
        try:
            with zipfile.ZipFile(archive_path) as archive:
                names=['/'+item.filename.replace('\\','/').lstrip('/') for item in archive.infolist()]
                if any('/printed_label_train_v1/train/normal/' in name for name in names):
                    matching.append(archive_path)
        except zipfile.BadZipFile:
            pass
    if len(matching)==1:
        extraction_root=WORK/'uploaded_data'
        extraction_root.mkdir(parents=True,exist_ok=True)
        resolved=extraction_root.resolve()
        with zipfile.ZipFile(matching[0]) as archive:
            for item in archive.infolist():
                target=(extraction_root/item.filename).resolve()
                if target != resolved and resolved not in target.parents:
                    raise ValueError(f'Unsafe ZIP member: {item.filename}')
            archive.extractall(extraction_root)
        dataset_candidates=find_dataset(extraction_root)
        print('Extracted:',matching[0])
if len(dataset_candidates)!=1:
    raise FileNotFoundError('Expected one printed_label_train_v1 dataset, found: '+repr([str(p) for p in dataset_candidates]))
DATA_ROOT=dataset_candidates[0]

SEED = 230224
TARGET_STEPS = 2000
BATCH_SIZE = 2
SAVE_EVERY = 250
LEARNING_RATE = 1e-4
IMAGE_SIZE = 224
print('GPU:', torch.cuda.get_device_name(0))
print('Dataset:', DATA_ROOT)
print('Working output:', WORK)


In [ ]:
# Install only missing runtime packages. Do not install the upstream requirements file.
missing = []
for package, module in [('timm','timm'), ('einops','einops'), ('numba','numba')]:
    if importlib.util.find_spec(module) is None:
        missing.append(package)
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
print('Dependencies ready.')


In [ ]:
script = WORK / 'author_smoke.py'
script.write_text('"""Run a bounded integration check of the author DTU-Net and Tsimplex code.\n\nThis is not training for anomaly detection, not a reproduced result, and not a\nperformance comparison. Downloads only five source files from a pinned commit.\n"""\nfrom __future__ import annotations\nimport argparse\nimport ast\nimport hashlib\nimport importlib.util\nimport json\nimport random\nimport sys\nimport time\nimport types\nimport urllib.error\nimport urllib.request\nfrom pathlib import Path\n\nCOMMIT = \'dc4a9bd2a2a5b1c31223daab4bdfea3f6a5b2990\'\nBASE_URL = f\'https://raw.githubusercontent.com/MAXNORM8650/Annotsim/{COMMIT}/\'\nFILES = [\'src/models/UModels/UDHVT.py\',\'GaussianDiffusion.py\',\n         \'utils/Simplex/constants.py\',\'utils/Simplex/internals.py\',\'utils/Simplex/noise.py\']\nEXPECTED = {\n \'utils/Simplex/constants.py\':\'52bf6ba3e2c0d386fa420382de380093a8dd61f488765cb812b13e25d0be7294\',\n \'utils/Simplex/internals.py\':\'ef67562885dcfe3356acd97784fe10660bf21238be7bcc608e86053c529fd61a\',\n \'src/models/UModels/UDHVT.py\':\'f7303c4dd228a3f5e1ab98d16fe1db7c7abfecfa97f683e89449128c5a03a4c2\',\n \'GaussianDiffusion.py\':\'cdf7a2143a441d20a3250458c0683c53ac1484ef8f0d0927831e66a34b52ec9a\',\n \'utils/Simplex/noise.py\':\'d114b6898369a0299e48f95fe4165fb3d587dcb8d7e257b58aef02077b6249d3\',\n}\n\n\ndef fetch_sources(root):\n    hashes={}\n    for name in FILES:\n        destination=root/name\n        destination.parent.mkdir(parents=True,exist_ok=True)\n        if not destination.exists():\n            print(\'Downloading\',name,flush=True)\n            last_error=None\n            for attempt in range(1,4):\n                try:\n                    with urllib.request.urlopen(BASE_URL+name,timeout=45) as response:\n                        payload=response.read()\n                    break\n                except urllib.error.URLError as exc:\n                    last_error=exc\n                    print(f\'Network attempt {attempt}/3 failed: {exc}\',flush=True)\n                    if attempt < 3:time.sleep(2*attempt)\n            else:\n                raise RuntimeError(\n                    \'Could not download the pinned public author files. In Kaggle, \'\n                    \'open Settings, turn Internet on, then rerun this cell.\'\n                ) from last_error\n            if name in EXPECTED and hashlib.sha256(payload).hexdigest()!=EXPECTED[name]:\n                raise ValueError(f\'Inspected-source hash mismatch: {name}; stop and review this revision.\')\n            destination.write_bytes(payload)\n        digest=hashlib.sha256(destination.read_bytes()).hexdigest()\n        if name in EXPECTED and digest!=EXPECTED[name]:\n            raise ValueError(f\'Cached-source hash mismatch: {name}; use a fresh cache after review.\')\n        hashes[name]=digest\n    return hashes\n\n\ndef load_module(name,path):\n    spec=importlib.util.spec_from_file_location(name,path)\n    module=importlib.util.module_from_spec(spec)\n    sys.modules[name]=module\n    spec.loader.exec_module(module)\n    return module\n\n\ndef load_author_components(source_root,output):\n    import numpy as np\n    import torch\n    import torch.nn as nn\n    # Namespace isolation avoids importing the repository\'s unrelated experiments.\n    package=types.ModuleType(\'labelinspect_author_simplex\')\n    package.__path__=[str(source_root/\'utils/Simplex\')]\n    sys.modules[package.__name__]=package\n    noise=load_module(package.__name__+\'.noise\',source_root/\'utils/Simplex/noise.py\')\n\n    original=(source_root/\'src/models/UModels/UDHVT.py\').read_text(encoding=\'utf8\')\n    replacements={\n      \'from torchvision import models\':\'# Removed unused torchvision.models import.\',\n      \'from timm.data import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD, IMAGENET_INCEPTION_MEAN, IMAGENET_INCEPTION_STD\':\'# Removed unused timm image constants.\',\n      \'from timm.models.helpers import build_model_with_cfg, named_apply, adapt_input_conv\':\'from timm.models._manipulate import named_apply\',\n      \'from timm.models.layers import trunc_normal_, lecun_normal_, to_2tuple\':\'from timm.layers import trunc_normal_, lecun_normal_, to_2tuple\',\n      \'from timm.models.registry import register_model\':\'# Removed unused timm registry import.\',\n    }\n    for old,new in replacements.items():\n        if original.count(old)!=1:raise ValueError(\'Compatibility patch no longer matches the inspected source: \'+old)\n        original=original.replace(old,new)\n    patched=output/\'author_UDHVT_compat.py\'\n    patched.write_text(\'import numpy as np\\n\'+original,encoding=\'utf8\')\n    model_module=load_module(\'labelinspect_author_model\',patched)\n\n    # Keep the author definitions, including its variance convention. Avoid the\n    # top-level imports for unused image losses, plotting, datasets and backbones.\n    tree=ast.parse((source_root/\'GaussianDiffusion.py\').read_text(encoding=\'utf8\'))\n    wanted={\'get_beta_schedule\',\'extract\',\'mean_flat\',\'generate_simplex_4noise\',\'GaussianDiffusionModel\'}\n    selected=[node for node in tree.body if isinstance(node,(ast.FunctionDef,ast.ClassDef)) and node.name in wanted]\n    if {node.name for node in selected}!=wanted:raise ValueError(\'Required author diffusion definitions are missing\')\n    reduced=ast.Module(body=selected,type_ignores=[])\n    namespace={\'np\':np,\'torch\':torch,\'nn\':nn,\'OpenSimplex\':noise.OpenSimplex}\n    exec(compile(reduced,\'author_diffusion_l2_subset.py\',\'exec\'),namespace)\n    (output/\'author_diffusion_l2_subset.py\').write_text(ast.unparse(reduced),encoding=\'utf8\')\n\n    class NoisePredictionAdapter(nn.Module):\n        def __init__(self,backbone):super().__init__();self.backbone=backbone\n        def forward(self,x,t,y=None):\n            if y is not None:raise ValueError(\'This initial adapter supports the normal-only, unconditioned path\')\n            result=self.backbone(x,t,y=None)\n            prediction=result[0] if isinstance(result,tuple) else result\n            if prediction.shape!=x.shape:raise ValueError(\'Noise prediction does not match input shape\')\n            return prediction\n\n    return model_module,namespace,NoisePredictionAdapter,{\n      \'imports\':replacements,\'extra_import\':\'numpy for the author PositionalEmbedding helper\',\n      \'adapter\':\'Select tuple element 0; preserve the backbone computation.\',\n      \'diffusion_loading\':\'AST-load only the author definitions required for Gaussian/Tsimplex L2 and sampling; other losses are not supported.\',\n      \'sampling\':\'Pass denoise_fn=noise_fn so reverse steps use configured O/mu/p instead of the author alternate branch defaults.\',\n      \'calling_convention\':\'Set author diffusion train=False to select model(x,t,y=lab) during sampling. This is a dispatch flag; the model is explicitly switched with model.train()/eval().\',\n    }\n\n\ndef synthetic_batch(size,batch,device):\n    import numpy as np\n    import torch\n    from PIL import Image,ImageDraw,ImageFont\n    im=Image.new(\'L\',(size,size),235);draw=ImageDraw.Draw(im)\n    try:font=ImageFont.truetype(\'DejaVuSans.ttf\',20)\n    except OSError:font=ImageFont.load_default(size=20)\n    draw.rectangle((12,12,size-12,size-12),outline=20,width=2)\n    draw.text((24,45),\'LABEL A-104\',font=font,fill=20)\n    draw.text((24,90),\'BATCH 2026\',font=font,fill=20)\n    x=torch.from_numpy(np.asarray(im).copy()).float()/127.5-1\n    return x[None,None].repeat(batch,3,1,1).to(device)\n\n\ndef save_preview(x,reconstructed,destination):\n    from PIL import Image,ImageDraw\n    import numpy as np\n    images=[]\n    for tensor in [x,reconstructed]:\n        array=((tensor[0].detach().float().cpu().permute(1,2,0).numpy()+1)/2*255).clip(0,255).astype(np.uint8)\n        images.append(Image.fromarray(array))\n    sheet=Image.new(\'RGB\',(520,302),\'white\');draw=ImageDraw.Draw(sheet)\n    draw.text((12,10),\'INTEGRATION CHECK ONLY - TWO UPDATES\',fill=\'darkred\')\n    draw.text((12,32),\'Synthetic input\',fill=\'black\');draw.text((268,32),\'8-step reconstruction\',fill=\'black\')\n    for i,im in enumerate(images):sheet.paste(im.resize((224,224)),(12+i*256,54))\n    draw.text((12,283),\'No anomaly-removal or accuracy claim.\',fill=\'darkred\');sheet.save(destination)\n\n\ndef run(output,cache=None):\n    import importlib.metadata\n    import numpy as np\n    import torch\n    import numba\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    if not torch.cuda.is_available():raise RuntimeError(\'Select a GPU accelerator before running this notebook\')\n    cache=Path(cache) if cache else output/\'upstream\'/COMMIT\n    hashes=fetch_sources(cache)\n    random.seed(230224);np.random.seed(230224);torch.manual_seed(230224)\n    numba.set_num_threads(min(2,numba.get_num_threads()))\n    module,ns,adapter_type,patches=load_author_components(cache,output)\n    config={\'img_size\':224,\'patch_size\':16,\'in_chans\':3,\'embed_dim\':384,\'depth\':12,\n            \'num_heads\':6,\'mlp_ratio\':4.,\'num_classes\':None,\'mlp_time_embed\':True,\n            \'use_dec\':[\'DAFF\',\'DAFF\',\'DAFF\'],\'PE_type\':\'SPE\',\'refinement\':True,\'qkv_bias\':False}\n    report={\'status\':\'running\',\'scope\':\'integration_check_only\',\'upstream_commit\':COMMIT,\n            \'source_sha256\':hashes,\'compatibility_changes\':patches,\'model_configuration\':config,\n            \'configuration_note\':\'Illustrated SPE/DMHA/HFF/refinement variant. Code depth=12 gives six encoder blocks, one middle, six decoder blocks. This is not asserted to match every paper table.\',\n            \'torch\':torch.__version__,\'gpu\':torch.cuda.get_device_name(0),\n            \'gpu_vram_gib\':torch.cuda.get_device_properties(0).total_memory/2**30,\n            \'packages\':{n:importlib.metadata.version(n) for n in [\'timm\',\'einops\',\'numba\',\'numpy\']},\n            \'noise_parameters\':{\'octave\':6,\'frequency\':64,\'persistence\':.9},\n            \'training_steps\':2,\'batch_size\':2,\'total_diffusion_steps\':1000,\'reconstruction_steps\':8}\n    (output/\'integration_report.json\').write_text(json.dumps(report,indent=2))\n    device=torch.device(\'cuda:0\');torch.cuda.reset_peak_memory_stats()\n    print(\'Building author DTU-Net, width 384, six attention heads...\',flush=True)\n    backbone=module.UDHVT(**config).to(device);model=adapter_type(backbone)\n    report[\'parameter_count\']=sum(p.numel() for p in model.parameters())\n    x=synthetic_batch(224,2,device)\n    t=torch.tensor([50,150],device=device,dtype=torch.long)\n    diffusion=ns[\'GaussianDiffusionModel\']([224,224],ns[\'get_beta_schedule\'](1000,\'cosine\'),img_channels=3,\n                 loss_type=\'l2\',noise=\'4dsimplex\',octave=6,frequency=64,persistence=.9,train=False)\n    print(\'Compiling the author 4D noise function on CPU; first use may take a few minutes...\',flush=True)\n    start=time.perf_counter()\n    probe=torch.zeros(1,1,4,4,device=device)\n    diffusion.noise_fn(probe,torch.tensor([5],device=device))\n    report[\'noise_first_compile_seconds\']=time.perf_counter()-start\n    noise=diffusion.noise_fn(x,t).float()\n    assert noise.shape==x.shape and torch.isfinite(noise).all()\n    assert not torch.allclose(noise[0],noise[1]),\'Different time coordinates unexpectedly generated identical samples\'\n    report[\'noise_shape\']=list(noise.shape)\n    report[\'noise_mean\']=float(noise.mean());report[\'noise_std\']=float(noise.std())\n    report[\'noise_normalization\']=\'Author raw amplitude retained; no per-sample standardization.\'\n    report[\'batch_noise_note\']=\'The author generator uses t as the fourth coordinate. Duplicate time coordinates with one seed can produce identical noise across batch entries; this remains to be assessed during training.\'\n    model.train();optimizer=torch.optim.AdamW(model.parameters(),lr=1e-4,weight_decay=0.)\n    report[\'losses\']=[];report[\'gradient_norms\']=[]\n    tracked=backbone.pos_embed.detach().clone()\n    for step in range(2):\n        optimizer.zero_grad(set_to_none=True)\n        losses,noisy,predicted=diffusion.calc_loss(model,x,None,t)\n        loss=losses[\'loss\'].mean()\n        assert predicted.shape==x.shape and torch.isfinite(loss)\n        loss.backward()\n        grads=[p.grad for p in model.parameters() if p.grad is not None]\n        assert grads and all(torch.isfinite(g).all() for g in grads)\n        norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.)\n        optimizer.step()\n        report[\'losses\'].append(float(loss.detach()))\n        report[\'gradient_norms\'].append(float(norm))\n        print(f\'Update {step+1}/2 passed; L2 noise loss {float(loss.detach()):.6f}\',flush=True)\n    assert not torch.equal(tracked,backbone.pos_embed.detach()),\'Optimizer did not change the tracked parameter\'\n    report[\'tracked_parameter_changed\']=True\n    report[\'parameters_without_grad\']=[name for name,p in model.named_parameters() if p.grad is None]\n    model.eval()\n    print(\'Checking eight author reverse-diffusion steps. This model is not trained for detection.\',flush=True)\n    with torch.no_grad():\n        result=diffusion.forward_backward(model,x[:1],None,see_whole_sequence=None,t_distance=8,denoise_fn=\'noise_fn\')\n    assert result.shape==x[:1].shape and torch.isfinite(result).all()\n    residual=(x[:1]-result).square().mean(dim=1)\n    assert residual.shape==(1,224,224) and torch.isfinite(residual).all()\n    save_preview(x,result,output/\'integration_preview.png\')\n    report[\'reconstruction_shape\']=list(result.shape);report[\'residual_shape\']=list(residual.shape)\n    report[\'peak_gpu_allocated_gib\']=torch.cuda.max_memory_allocated()/2**30\n    report[\'peak_gpu_reserved_gib\']=torch.cuda.max_memory_reserved()/2**30\n    report[\'status\']=\'passed\'\n    report[\'not_completed\']=[\'Training a useful anomaly model\',\'Real label dataset\',\'Paper metrics reproduction\',\'Quality comparison with CPU baseline\']\n    (output/\'integration_report.json\').write_text(json.dumps(report,indent=2))\n    print(\'\\nDTU-NET + TSIMPLEX INTEGRATION CHECK PASSED\',flush=True)\n    print(json.dumps({k:report[k] for k in [\'parameter_count\',\'noise_shape\',\'losses\',\'reconstruction_shape\',\'peak_gpu_allocated_gib\',\'peak_gpu_reserved_gib\']},indent=2))\n    print(\'Saved:\',output/\'integration_report.json\',flush=True)\n    return report\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--output\',default=\'artifacts/author_integration\')\n    parser.add_argument(\'--cache\',default=None)\n    args=parser.parse_args();run(args.output,args.cache)\n', encoding='utf8')
print('Wrote:', script)


In [ ]:
# Load the same pinned and hash-verified author components that passed Stage 2.
spec = importlib.util.spec_from_file_location('labelinspect_author_smoke', WORK/'author_smoke.py')
author = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = author
spec.loader.exec_module(author)

import numba
numba.set_num_threads(min(2, numba.get_num_threads()))
source_cache = WORK/'upstream'/author.COMMIT
hashes = author.fetch_sources(source_cache)
model_module, diffusion_ns, Adapter, compatibility = author.load_author_components(source_cache, OUTPUT)
print('Pinned author commit:', author.COMMIT)


In [ ]:
IMAGE_EXTENSIONS={'.jpg','.jpeg','.png'}
train_paths=sorted(
    path for path in (DATA_ROOT/'train'/'normal').iterdir()
    if path.suffix.lower() in IMAGE_EXTENSIONS
)
assert len(train_paths)==40, f'Expected 40 registered normal images, found {len(train_paths)}'
physical_ids={path.stem.split('_')[0] for path in train_paths}
view_ids={path.stem.split('_')[1] for path in train_paths}
assert physical_ids=={f'N{index:02d}' for index in range(1,9)}, physical_ids
assert view_ids=={f'B{index}' for index in range(1,6)}, view_ids
print('Training normals:',len(train_paths))
print('Physical labels:',sorted(physical_ids))
print('Capture groups:',sorted(view_ids))

RESAMPLE=getattr(Image,'Resampling',Image).BILINEAR

def pad_to_square(image,size=IMAGE_SIZE):
    image=ImageOps.contain(image,(size,size),RESAMPLE)
    canvas=Image.new('RGB',(size,size),(238,238,238))
    canvas.paste(image,((size-image.width)//2,(size-image.height)//2))
    return canvas

def load_training_image(path,rng):
    with Image.open(path) as source:
        image=source.convert('RGB')
        # Mild deterministic jitter represents residual registration and lighting variation.
        image=ImageEnhance.Brightness(image).enhance(float(rng.uniform(0.90,1.10)))
        image=ImageEnhance.Contrast(image).enhance(float(rng.uniform(0.92,1.08)))
        angle=float(rng.uniform(-1.0,1.0))
        image=image.rotate(angle,resample=Image.Resampling.BICUBIC,expand=False,fillcolor=(238,238,238))
        image=pad_to_square(image)
        array=np.asarray(image,dtype=np.float32).copy()/127.5-1.0
    return torch.from_numpy(array).permute(2,0,1)

def deterministic_batch(step):
    rng=np.random.default_rng(np.random.SeedSequence([SEED,step]))
    indices=rng.choice(len(train_paths),size=BATCH_SIZE,replace=False)
    return torch.stack([load_training_image(train_paths[int(i)],rng) for i in indices])

def deterministic_times(step):
    rng=np.random.default_rng(np.random.SeedSequence([SEED,step,1]))
    values=rng.integers(1,1000,size=BATCH_SIZE,dtype=np.int64)
    if len(set(values.tolist()))!=len(values):
        values[1]=(values[0]%999)+1
    return torch.from_numpy(values)

sample=deterministic_batch(0)
assert sample.shape==(BATCH_SIZE,3,IMAGE_SIZE,IMAGE_SIZE)
assert torch.isfinite(sample).all() and sample.min()>=-1 and sample.max()<=1
print('Training batch check:',tuple(sample.shape),float(sample.min()),float(sample.max()))


In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda:0')
model_config = {
    'img_size':224, 'patch_size':16, 'in_chans':3, 'embed_dim':384,
    'depth':12, 'num_heads':6, 'mlp_ratio':4., 'num_classes':None,
    'mlp_time_embed':True, 'use_dec':['DAFF','DAFF','DAFF'],
    'PE_type':'SPE', 'refinement':True, 'qkv_bias':False,
}
backbone = model_module.UDHVT(**model_config).to(device)
model = Adapter(backbone)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.)
diffusion = diffusion_ns['GaussianDiffusionModel'](
    [224,224], diffusion_ns['get_beta_schedule'](1000,'cosine'),
    img_channels=3, loss_type='l2', noise='4dsimplex',
    octave=6, frequency=64, persistence=.9, train=False,
)

latest = CHECKPOINT_DIR/'printed_label_latest.pt'
attached=list(KAGGLE_INPUT.rglob('printed_label_latest.pt'))
if len(attached)>1:
    raise ValueError('Attach at most one printed_label_latest.pt checkpoint.')
if len(attached)==1:
    shutil.copy2(attached[0],latest)

start_step=0
history=[]
if latest.is_file():
    checkpoint=torch.load(latest,map_location=device,weights_only=False)
    if checkpoint.get('author_commit')!=author.COMMIT:
        raise ValueError('Checkpoint author commit does not match this notebook.')
    if checkpoint.get('dataset_category')!='printed_label_train_v1':
        raise ValueError('This is not a printed-label checkpoint.')
    model.load_state_dict(checkpoint['model'])
    optimizer.load_state_dict(checkpoint['optimizer'])
    start_step=int(checkpoint['step'])
    history=list(checkpoint.get('history',[]))
    print('Resuming from step',start_step)
else:
    print('Starting a new printed-label model from step 0.')
if start_step>TARGET_STEPS:
    raise ValueError(f'Checkpoint is at step {start_step}; set TARGET_STEPS to at least that value.')

probe=torch.zeros(1,1,4,4,device=device)
diffusion.noise_fn(probe,torch.tensor([5],device=device))
torch.cuda.reset_peak_memory_stats()


In [ ]:
def save_checkpoint(step):
    payload={
        'step':step,
        'model':model.state_dict(),
        'optimizer':optimizer.state_dict(),
        'history':history,
        'author_commit':author.COMMIT,
        'source_sha256':hashes,
        'model_config':model_config,
        'noise_parameters':{'octave':6,'frequency':64,'persistence':0.9},
        'seed':SEED,
        'dataset_category':'printed_label_train_v1',
        'protocol':'40 registered normal photos from eight physical labels; no test images used',
        'preprocessing':'four-fiducial perspective registration; aspect-preserving resize and padding',
    }
    temporary=CHECKPOINT_DIR/'printed_label_latest.tmp.pt'
    torch.save(payload,temporary)
    temporary.replace(latest)

model.train()
run_started=time.perf_counter()
step_times=[]
for step in range(start_step,TARGET_STEPS):
    step_started=time.perf_counter()
    x=deterministic_batch(step).to(device,non_blocking=True)
    t=deterministic_times(step).to(device)
    optimizer.zero_grad(set_to_none=True)
    losses,noisy,predicted=diffusion.calc_loss(model,x,None,t)
    loss=losses['loss'].mean()
    if not torch.isfinite(loss):
        raise FloatingPointError(f'Non-finite loss at step {step+1}: {loss}')
    loss.backward()
    grad_norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
    if not torch.isfinite(grad_norm):
        raise FloatingPointError(f'Non-finite gradient norm at step {step+1}')
    optimizer.step()
    elapsed=time.perf_counter()-step_started
    step_times.append(elapsed)
    history.append({'step':step+1,'loss':float(loss.detach()),'grad_norm':float(grad_norm),'seconds':elapsed})
    if (step+1)%10==0 or step==start_step:
        recent=np.mean([item['loss'] for item in history[-10:]])
        print(f'Step {step+1:4d}/{TARGET_STEPS} | loss {float(loss):.6f} | recent mean {recent:.6f} | {elapsed:.2f}s')
    if (step+1)%SAVE_EVERY==0 or step+1==TARGET_STEPS:
        save_checkpoint(step+1)

run_seconds=time.perf_counter()-run_started
print('Checkpoint:',latest)
print('Steps completed this run:',TARGET_STEPS-start_step)


In [ ]:
import csv
import matplotlib.pyplot as plt

with (OUTPUT/'training_history.csv').open('w',newline='') as stream:
    writer=csv.DictWriter(stream,fieldnames=['step','loss','grad_norm','seconds'])
    writer.writeheader(); writer.writerows(history)

loss_values=[item['loss'] for item in history]
window=min(20,len(loss_values))
smoothed=np.convolve(loss_values,np.ones(window)/window,mode='valid') if window else []
fig,ax=plt.subplots(figsize=(9,4))
ax.plot(range(1,len(loss_values)+1),loss_values,alpha=.35,label='step loss')
if len(smoothed):
    ax.plot(range(window,len(loss_values)+1),smoothed,linewidth=2,label=f'{window}-step mean')
ax.set(xlabel='Optimizer step',ylabel='L2 noise-prediction loss',title='Printed-label normal-only training')
ax.grid(alpha=.25); ax.legend(); fig.tight_layout()
fig.savefig(OUTPUT/'loss_curve.png',dpi=160)
plt.show()

report={
    'status':'passed',
    'scope':'printed_label_training_only',
    'paper_result_reproduced':False,
    'dataset':'LabelInspect real printed labels v1',
    'train_normal_count':len(train_paths),
    'train_physical_label_count':len(physical_ids),
    'validation_images_used':0,
    'test_images_used':0,
    'author_commit':author.COMMIT,
    'model_parameters':sum(p.numel() for p in model.parameters()),
    'target_steps':TARGET_STEPS,
    'start_step':start_step,
    'steps_completed_this_run':TARGET_STEPS-start_step,
    'initial_loss':history[0]['loss'] if history else None,
    'final_loss':history[-1]['loss'] if history else None,
    'last_20_mean_loss':float(np.mean(loss_values[-20:])) if loss_values else None,
    'median_step_seconds_this_run':float(np.median(step_times)) if step_times else None,
    'run_seconds':run_seconds,
    'peak_gpu_allocated_gib':torch.cuda.max_memory_allocated()/2**30,
    'peak_gpu_reserved_gib':torch.cuda.max_memory_reserved()/2**30,
    'gpu':torch.cuda.get_device_name(0),
    'checkpoint':str(latest),
    'next_decision':'Collect N09-N12 and defective labels, register them, calibrate using validation normals, then run one locked test.',
}
(OUTPUT/'training_report.json').write_text(json.dumps(report,indent=2))
shutil.copy2(latest,Path('/kaggle/working/printed_label_latest.pt'))
archive=shutil.make_archive('/kaggle/working/printed_label_training_results','zip',OUTPUT)
print(json.dumps(report,indent=2))
print('\nPRINTED-LABEL TRAINING PASSED')
print('Download checkpoint: /kaggle/working/printed_label_latest.pt')
print('Download result bundle:',archive)
